In [ ]:
"""
PHISHING DETECTION - Neural Network with Class Weights
Multi-class Classification: phishing, benign, defacement, malware
Heavy regularization + Class weights to handle imbalanced data
Character-level CNN with high dropout and L2 regularization
"""

# ============================================
# SETUP & IMPORTS
# ============================================

!pip install -q scikit-learn

import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import pickle
import gc
import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

# Memory optimization
import os
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'

# ============================================
# 1. LOAD PRE-SPLIT DATA FROM KAGGLE
# ============================================

DATASET_PATH = '/kaggle/input/1-8mil-phishing-detection-system-dataset'

print("📥 Loading pre-split dataset from Kaggle...")

try:
    train_df = pd.read_csv(DATASET_PATH + 'train.csv')
    val_df = pd.read_csv(DATASET_PATH + 'validation.csv')
    test_df = pd.read_csv(DATASET_PATH + 'test.csv')
    print("✅ Loaded from Kaggle dataset")
except:
    print("⚠️ Kaggle path not found, using file upload...")
    from google.colab import files
    uploaded = files.upload()
    
    train_df = pd.read_csv('train.csv')
    val_df = pd.read_csv('validation.csv')
    test_df = pd.read_csv('test.csv')
    print("✅ Loaded from uploaded files")

# ============================================
# 2. CONVERT LABELS TO NUMERIC
# ============================================

print(f"\n🔄 Converting labels to numeric...")

label_map = {'benign': 0, 'defacement': 1, 'malware': 2, 'phishing': 3}
label_names = ['benign', 'defacement', 'malware', 'phishing']

def convert_label(label):
    if isinstance(label, str):
        return label_map.get(label.lower(), 0)
    else:
        return int(label)

train_df['label'] = train_df['label'].apply(convert_label)
val_df['label'] = val_df['label'].apply(convert_label)
test_df['label'] = test_df['label'].apply(convert_label)

print(f"\n📊 Dataset splits:")
print(f"   Train: {len(train_df):,}")
print(f"   Val:   {len(val_df):,}")
print(f"   Test:  {len(test_df):,}")

print(f"\n📊 Label distribution:")
for i, name in enumerate(label_names):
    count = (train_df['label'] == i).sum()
    pct = (count / len(train_df) * 100)
    print(f"   {name}: {count:,} ({pct:.2f}%)")

# ============================================
# 3. CALCULATE CLASS WEIGHTS
# ============================================

print("\n" + "="*70)
print("⚖️ CALCULATING CLASS WEIGHTS FOR BALANCED TRAINING")
print("="*70)

# Compute class weights to handle imbalanced dataset
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_df['label']),
    y=train_df['label']
 )

class_weights = {i: weight for i, weight in enumerate(class_weights_array)}

print(f"\n📊 Class Weights (higher weight = more penalty for misclassification):")
for i, name in enumerate(label_names):
    count = (train_df['label'] == i).sum()
    print(f"   {name:12}: {class_weights[i]:.4f}x  ({count:,} samples)")

print(f"\n✅ Class weights will help the model learn all classes equally well,")
print(f"   even if some classes have fewer training examples.")

# ============================================
# 4. BUILD CHARACTER VOCABULARY
# ============================================

print("\n🔤 Building character vocabulary...")

all_chars = set()
for url in tqdm(train_df['url'], desc="Analyzing URLs"):
    all_chars.update(url)

chars = sorted(list(all_chars))
char_to_idx = {ch: i+1 for i, ch in enumerate(chars)}
idx_to_char = {i+1: ch for i, ch in enumerate(chars)}
vocab_size = len(chars) + 1
MAX_URL_LEN = 200

print(f"✅ Vocabulary size: {vocab_size}")

# Save char mapping for later use
char_mapping = {
    'char_to_idx': char_to_idx,
    'idx_to_char': idx_to_char,
    'vocab_size': vocab_size,
    'max_url_len': MAX_URL_LEN
}

with open('char_to_idx.pkl', 'wb') as f:
    pickle.dump(char_mapping, f)
print("💾 Saved: char_to_idx.pkl")

# ============================================
# 5. ENCODE URLs (CHARACTER-LEVEL)
# ============================================

def encode_url(url):
    return [char_to_idx.get(c, 0) for c in url[:MAX_URL_LEN]]

print(f"\n🔧 Encoding URLs (max length: {MAX_URL_LEN})...")

def encode_batch(df, batch_size=10000):
    encoded = []
    for i in tqdm(range(0, len(df), batch_size), desc="Encoding"):
        batch = df['url'].iloc[i:i+batch_size]
        encoded.extend([encode_url(url) for url in batch])
    return encoded

X_train_encoded = encode_batch(train_df)
X_val_encoded = encode_batch(val_df)
X_test_encoded = encode_batch(test_df)

print("📦 Padding sequences...")
X_train = keras.preprocessing.sequence.pad_sequences(X_train_encoded, maxlen=MAX_URL_LEN, padding='post')
X_val = keras.preprocessing.sequence.pad_sequences(X_val_encoded, maxlen=MAX_URL_LEN, padding='post')
X_test = keras.preprocessing.sequence.pad_sequences(X_test_encoded, maxlen=MAX_URL_LEN, padding='post')

y_train = train_df['label'].values
y_val = val_df['label'].values
y_test = test_df['label'].values

del X_train_encoded, X_val_encoded, X_test_encoded, train_df, val_df, test_df
gc.collect()

print(f"\n✅ Encoding complete!")
print(f"   X_train: {X_train.shape}")
print(f"   X_val:   {X_val.shape}")
print(f"   X_test:  {X_test.shape}")

# ============================================
# 6. BUILD NEURAL NETWORK (ANTI-MEMORIZATION)
# ============================================

print("\n" + "="*70)
print("🧠 BUILDING NEURAL NETWORK - ANTI-MEMORIZATION ARCHITECTURE")
print("="*70)

print("\n🛡️ Regularization Techniques:")
print("   ✅ High Dropout (0.6-0.8) - Prevents memorization")
print("   ✅ L2 Regularization - Penalizes large weights")
print("   ✅ Batch Normalization - Stabilizes learning")
print("   ✅ Gaussian Noise - Adds randomness to prevent overfitting")
print("   ✅ Early Stopping - Stops when validation stops improving")
print("   ✅ Class Weights - Balances learning across all classes")

# L2 regularization strength
l2_reg = 0.001

# Functional model for a stronger architecture
inputs = keras.Input(shape=(MAX_URL_LEN,), dtype='int32')
x = layers.Embedding(
    vocab_size,
    128,
    input_length=MAX_URL_LEN,
    embeddings_regularizer=regularizers.l2(l2_reg)
 )(inputs)

# Noise + dropout to fight memorization
x = layers.GaussianNoise(0.1)(x)
x = layers.SpatialDropout1D(0.2)(x)

# Multi-kernel convolution (captures short/medium/long patterns)
b1 = layers.Conv1D(128, 3, activation='relu', padding='same',
                   kernel_regularizer=regularizers.l2(l2_reg))(x)
b2 = layers.Conv1D(128, 5, activation='relu', padding='same',
                   kernel_regularizer=regularizers.l2(l2_reg))(x)
b3 = layers.Conv1D(128, 7, activation='relu', padding='same',
                   kernel_regularizer=regularizers.l2(l2_reg))(x)
x = layers.Concatenate()([b1, b2, b3])
x = layers.BatchNormalization()(x)
x = layers.MaxPooling1D(2)(x)
x = layers.Dropout(0.6)(x)

# Residual separable conv block (deeper features, fewer params)
res = layers.Conv1D(256, 1, padding='same',
                    kernel_regularizer=regularizers.l2(l2_reg))(x)
x = layers.SeparableConv1D(256, 3, activation='relu', padding='same',
                           depthwise_regularizer=regularizers.l2(l2_reg),
                           pointwise_regularizer=regularizers.l2(l2_reg))(x)
x = layers.BatchNormalization()(x)
x = layers.SeparableConv1D(256, 3, activation='relu', padding='same',
                           depthwise_regularizer=regularizers.l2(l2_reg),
                           pointwise_regularizer=regularizers.l2(l2_reg))(x)
x = layers.Add()([x, res])
x = layers.MaxPooling1D(2)(x)
x = layers.Dropout(0.6)(x)

# BiGRU for sequence modeling
x = layers.Bidirectional(
    layers.GRU(
        128,
        return_sequences=True,
        dropout=0.3,
        recurrent_dropout=0.3,
        kernel_regularizer=regularizers.l2(l2_reg),
        recurrent_regularizer=regularizers.l2(l2_reg)
    )
 )(x)

# Simple attention pooling
attn = layers.Dense(1, activation='tanh')(x)
attn = layers.Flatten()(attn)
attn = layers.Activation('softmax')(attn)
attn = layers.RepeatVector(x.shape[-1])(attn)
attn = layers.Permute([2, 1])(attn)
x = layers.Multiply()([x, attn])
x = layers.Lambda(lambda t: tf.reduce_sum(t, axis=1))(x)
x = layers.Dropout(0.7)(x)

# Dense layers with heavy regularization
x = layers.Dense(512, activation='relu', kernel_regularizer=regularizers.l2(l2_reg))(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.8)(x)

x = layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(l2_reg))(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.7)(x)

x = layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(l2_reg))(x)
x = layers.Dropout(0.7)(x)

# Output layer - 4 classes
outputs = layers.Dense(4, activation='softmax')(x)

model = keras.Model(inputs, outputs)

# Compile with learning rate scheduling
initial_lr = 0.001
lr_schedule = keras.optimizers.schedules.ExponentialDecay(
    initial_lr,
    decay_steps=5000,
    decay_rate=0.9,
    staircase=True
 )

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=lr_schedule),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
 )

print("\n🔧 Model Architecture:")
model.summary()

# ============================================
# 7. CALLBACKS FOR TRAINING
# ============================================

callbacks = [
    # Early stopping - prevents overfitting
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=7,  # More patience
        restore_best_weights=True,
        mode='max',
        verbose=1
    ),
    
    # Reduce learning rate when stuck
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=4,
        min_lr=1e-7,
        verbose=1
    ),
    
    # Save best model
    keras.callbacks.ModelCheckpoint(
        'best_model.keras',
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    )
]

# ============================================
# 8. TRAIN THE MODEL WITH CLASS WEIGHTS
# ============================================

print("\n" + "="*70)
print("🚀 TRAINING NEURAL NETWORK WITH CLASS WEIGHTS")
print("="*70)

BATCH_SIZE = 256
EPOCHS = 25

print(f"\n⚙️ Training Configuration:")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Max Epochs: {EPOCHS}")
print(f"   Initial Learning Rate: {initial_lr}")
print(f"   Regularization: L2={l2_reg}, Dropout=0.6-0.8, Gaussian Noise=0.1")
print(f"   Class Weights: Enabled (balances imbalanced classes)")

history = model.fit(
    X_train, y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(X_val, y_val),
    class_weight=class_weights,  # ← CLASS WEIGHTS FOR BALANCED LEARNING
    callbacks=callbacks,
    verbose=1
)

# ============================================
# 9. EVALUATE THE MODEL
# ============================================

print("\n" + "="*70)
print("📊 MODEL EVALUATION")
print("="*70)

# Load best model
best_model = keras.models.load_model('best_model.keras')

# Predictions
val_pred = best_model.predict(X_val, batch_size=BATCH_SIZE).argmax(axis=1)
test_pred = best_model.predict(X_test, batch_size=BATCH_SIZE).argmax(axis=1)

# Calculate metrics
val_acc = accuracy_score(y_val, val_pred)
val_f1 = f1_score(y_val, val_pred, average='weighted')
test_acc = accuracy_score(y_test, test_pred)
test_f1 = f1_score(y_test, test_pred, average='weighted')

print(f"\n📈 Results:")
print(f"   Validation Accuracy: {val_acc:.4f} ({val_acc*100:.2f}%)")
print(f"   Validation F1-Score: {val_f1:.4f}")
print(f"   Test Accuracy:       {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"   Test F1-Score:       {test_f1:.4f}")

# Per-class performance
print(f"\n📊 Per-Class Accuracy (Test Set):")
for i, name in enumerate(label_names):
    class_mask = (y_test == i)
    class_acc = (test_pred[class_mask] == i).sum() / class_mask.sum() if class_mask.sum() > 0 else 0
    print(f"   {name:12}: {class_acc:.4f} ({class_mask.sum():,} samples)")

# Detailed classification report
print(f"\n📋 Detailed Classification Report (Test Set):")
print(classification_report(y_test, test_pred, target_names=label_names, digits=4))

# ============================================
# 10. SAVE MODEL PROPERLY WITH VERIFICATION
# ============================================

print("\n" + "="*70)
print("💾 SAVING MODEL AND VERIFYING")
print("="*70)

# Save the final model
model.save('neural_network_model.keras')
print("✅ Saved: neural_network_model.keras")

# Save best model copy
best_model.save('neural_network_model_best.keras')
print("✅ Saved: neural_network_model_best.keras")

# CRITICAL: Verify the model works with different URLs
print("\n🧪 VERIFICATION TEST - Testing with different URLs...")

test_urls = [
    'https://www.google.com',
    'https://github.com',
    'http://phishing-login-verify-account-suspended.com',
    'http://192.168.1.1/malware.exe',
    'http://malicious-site-download-virus.net'
]

test_encoded = keras.preprocessing.sequence.pad_sequences(
    [[char_to_idx.get(c, 0) for c in url] for url in test_urls],
    maxlen=MAX_URL_LEN, padding='post'
)

# Test with saved model
loaded_model = keras.models.load_model('neural_network_model.keras')
predictions = loaded_model.predict(test_encoded, verbose=0)

print("\n📊 Verification Results:")
all_same = True
prev_pred = None

for i, (url, pred) in enumerate(zip(test_urls, predictions)):
    pred_class = label_names[np.argmax(pred)]
    confidence = np.max(pred) * 100
    print(f"\n   URL {i+1}: {url[:60]}")
    print(f"   → Prediction: {pred_class} ({confidence:.1f}%)")
    print(f"   → Probabilities: Benign={pred[0]:.3f}, Defacement={pred[1]:.3f}, Malware={pred[2]:.3f}, Phishing={pred[3]:.3f}")
    
    if prev_pred is not None:
        if not np.allclose(pred, prev_pred):
            all_same = False
    prev_pred = pred

if all_same:
    print("\n❌ WARNING: All predictions are IDENTICAL!")
    print("   Model may have broken weights. Check training logs.")
else:
    print("\n✅✅✅ SUCCESS! Model produces DIFFERENT predictions for different URLs!")
    print("   Model is working correctly!")

# Special test for famous brands
print("\n🏢 Testing Famous Brands (Should predict BENIGN):")
brand_urls = [
    'https://www.google.com',
    'https://www.facebook.com',
    'https://www.microsoft.com',
    'https://www.amazon.com'
 ]

brand_encoded = keras.preprocessing.sequence.pad_sequences(
    [[char_to_idx.get(c, 0) for c in url] for url in brand_urls],
    maxlen=MAX_URL_LEN, padding='post'
)

brand_predictions = loaded_model.predict(brand_encoded, verbose=0)

benign_count = 0
for url, pred in zip(brand_urls, brand_predictions):
    pred_class = label_names[np.argmax(pred)]
    benign_prob = pred[0]
    if pred_class == 'benign':
        benign_count += 1
    print(f"   {url:30} → {pred_class:10} (benign: {benign_prob:.2f})")

if benign_count >= 3:
    print(f"\n✅ Model correctly identifies {benign_count}/4 famous brands as benign!")
else:
    print(f"\n⚠️ Model only identifies {benign_count}/4 famous brands as benign")
    print("   May need more benign URL diversity or longer training")

# ============================================
# 11. VISUALIZATIONS
# ============================================

print("\n📊 Creating visualizations...")

# Training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Training History (Balanced Dataset with Class Weights)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Val Loss', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Loss History')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
print("✅ Saved: training_history.png")
plt.show()

# Confusion matrix
cm = confusion_matrix(y_test, test_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True,
            xticklabels=label_names, yticklabels=label_names)
plt.title('Confusion Matrix - Test Set', fontsize=16)
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
print("✅ Saved: confusion_matrix.png")
plt.show()

# ============================================
# 12. DOWNLOAD FILES FROM KAGGLE
# ============================================

print("\n" + "="*70)
print("📥 DOWNLOADING FILES FROM KAGGLE")
print("="*70)

files_to_download = [
    'neural_network_model.keras',
    'neural_network_model_best.keras',
    'char_to_idx.pkl',
    'training_history.png',
    'confusion_matrix.png'
 ]

try:
    from google.colab import files as colab_files
    print("\n📦 Downloading files...")
    for filename in files_to_download:
        try:
            colab_files.download(filename)
            print(f"   ✅ Downloaded: {filename}")
        except Exception as e:
            print(f"   ⚠️ Could not download {filename}: {e}")
except:
    print("\n✅ Files saved locally (not on Colab)")
    print("\n📁 Saved files:")
    for filename in files_to_download:
        print(f"   • {filename}")

# ============================================
# FINAL SUMMARY
# ============================================

print("\n" + "="*70)
print("✅ TRAINING COMPLETE!")
print("="*70)

total_urls = X_train.shape[0] + X_val.shape[0] + X_test.shape[0]
print(f"\n📊 Dataset: {total_urls:,} URLs (BALANCED)")
print(f"📈 Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"📈 Test F1-Score: {test_f1:.4f}")

print(f"\n📊 Per-Class Performance:")
for i, name in enumerate(label_names):
    class_mask = (y_test == i)
    class_acc = (test_pred[class_mask] == i).sum() / class_mask.sum() if class_mask.sum() > 0 else 0
    class_count = class_mask.sum()
    print(f"   {name:12}: {class_acc:.4f} ({class_count:,} samples)")

print(f"\n🛡️ Regularization Applied:")
print(f"   • Dropout: 0.6-0.8 (prevents memorization)")
print(f"   • L2 Regularization: {l2_reg}")
print(f"   • Gaussian Noise: 0.1")
print(f"   • Batch Normalization: Yes")
print(f"   • Early Stopping: Yes (patience=7)")
print(f"   • Class Weights: Yes (balances imbalanced classes)")
print(f"   • Balanced Dataset: Yes (1:1 benign:phishing ratio)")

print(f"\n📦 Files to download:")
for filename in files_to_download:
    print(f"   • {filename}")

print("\n" + "="*70)
print("🚀 Upload these files to your project:")
print("   1. best_model.keras → results_1mil550k_dataset/")
print("   2. char_to_idx.pkl → results_1mil550k_dataset/")
print("   3. Then run: streamlit run app.py")
print("="*70)